# RAGAS Evaluation — Stylists.ai RAG Pipeline

Evaluates retrieval quality by experimenting with chunking parameters and retrieval strategies.

## Steps
1. Environment setup
2. Load fashion knowledge base
3. Load synthetic test set from cache (SDG was run separately)
4. **Chunking experiments** — find optimal chunk_size and overlap
5. **Retrieval experiments** — naive, BM25, rerank, parent-child, ensemble
6. Compare all strategies with RAGAS metrics

## 1. Environment Setup

In [1]:
import os
import sys
import time

import nest_asyncio
import pandas as pd
from dotenv import load_dotenv

nest_asyncio.apply()

# The Jupyter kernel cwd is evals/ — go up one level to get project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# Enable LangSmith tracing for cost/latency analysis
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "stylists-ai-ragas-eval"

EVALS_DIR = os.path.join(PROJECT_ROOT, "evals")
RESULTS_DIR = os.path.join(EVALS_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"sys.path[0]: {sys.path[0]}")
print(f"rag exists: {os.path.isdir(os.path.join(PROJECT_ROOT, 'rag'))}")
print("Environment ready")

Project root: /Users/yingzheli/aimakerspace/stylists-ai-backend-cert
sys.path[0]: /Users/yingzheli/aimakerspace/stylists-ai-backend-cert
rag exists: True
Environment ready


## 2. Load Knowledge Base

In [2]:
from rag.loader import load_knowledge_files

docs = load_knowledge_files()
print(f"Loaded {len(docs)} documents")
print(f"Domains: {set(d.metadata.get('domain', 'unknown') for d in docs)}")

Loaded 24 documents
Domains: {'fundamentals', 'wardrobe_building', 'body_shapes', 'style_archetypes', 'occasion_dressing', 'color_theory'}


## 3. Load Synthetic Test Set (from cache)

SDG was run separately via `run_baseline_eval.py`. The test set is cached in
`synthetic_testset.csv` — 21 samples with balanced query types:
- 7 single-hop specific
- 7 multi-hop abstract  
- 7 multi-hop specific

In [3]:
test_df = pd.read_csv(os.path.join(EVALS_DIR, "synthetic_testset.csv"))
print(f"Loaded {len(test_df)} test samples")
print(f"\nQuery type distribution:")
print(test_df["synthesizer_name"].value_counts().to_string())
print(f"\nSample questions:")
for _, row in test_df[["user_input"]].head(5).iterrows():
    print(f"  - {row['user_input'][:100]}")

Loaded 21 test samples

Query type distribution:
synthesizer_name
single_hop_specific_query_synthesizer    7
multi_hop_abstract_query_synthesizer     7
multi_hop_specific_query_synthesizer     7

Sample questions:
  - How does geometry influence the visual harmony and proportional strategies recommended for a Deep Au
  - How can a beginner use leather in their outfits to create a stylish look, and what role does leather
  - As a Deep Autumn professional with an inverted triangle body shape who prefers classic, structured c
  - wat is h shap for mens?
  - What hourglass silhouette mean and how I know if I got it?


## 4. Shared Components

RAG prompt, LLM, evaluator, and helper function used across all experiments.

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas.llms import LangchainLLMWrapper
from ragas import EvaluationDataset, RunConfig
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
)
from ragas import evaluate as ragas_evaluate

# All experiments use gpt-4.1-mini
rag_llm = ChatOpenAI(model="gpt-4.1-mini")
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

RAG_PROMPT = """You are a personal AI stylist. Answer the question using ONLY the provided context.

### Question
{question}

### Context
{context}
"""
rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

METRICS = [
    LLMContextRecall(),
    Faithfulness(),
    FactualCorrectness(),
    ResponseRelevancy(),
    ContextEntityRecall(),
]


def run_experiment(retriever_fn, name, sleep_between=0):
    """Run a retriever through all test queries and score with RAGAS.

    Args:
        retriever_fn: Function that takes a question string and returns List[Document]
        name: Display name for this experiment
        sleep_between: Seconds to sleep between queries (for rate limiting)

    Returns:
        DataFrame with per-query RAGAS scores
    """
    print(f"\n{'='*60}")
    print(f"Running: {name}")
    print(f"{'='*60}")

    samples = []
    for i, (_, row) in enumerate(test_df.iterrows()):
        retrieved = retriever_fn(row["user_input"])
        docs_content = "\n\n".join(doc.page_content for doc in retrieved)
        messages = rag_prompt.format_messages(question=row["user_input"], context=docs_content)
        response = rag_llm.invoke(messages).content

        samples.append(SingleTurnSample(
            user_input=row["user_input"],
            response=response,
            reference=row["reference"],
            retrieved_contexts=[doc.page_content for doc in retrieved],
        ))
        if sleep_between > 0:
            time.sleep(sleep_between)
        if (i + 1) % 7 == 0:
            print(f"  Processed {i + 1}/{len(test_df)} queries")

    print(f"  Done with {len(samples)} queries. Evaluating with RAGAS...")

    result = ragas_evaluate(
        dataset=EvaluationDataset(samples=samples),
        metrics=METRICS,
        llm=evaluator_llm,
        run_config=RunConfig(timeout=360),
    )

    result_df = result.to_pandas()
    print(f"\n  Results for {name}:")
    for col in result_df.select_dtypes(include="number").columns:
        valid = result_df[col].dropna()
        print(f"    {col:40s} {valid.mean():.4f}  ({len(valid)}/{len(result_df)} valid)")

    return result_df


print("Shared components ready")

Shared components ready


/var/folders/r4/cqrwpld94ldc4rgfhq1fbx8h0000gn/T/ipykernel_70969/1869784151.py:8: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
/var/folders/r4/cqrwpld94ldc4rgfhq1fbx8h0000gn/T/ipykernel_70969/1869784151.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/var/folders/r4/cqrwpld94ldc4rgfhq1fbx8h0000gn/T/ipykernel_70969/1869784151.py:8: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import (

---
## 5. Chunking Experiments

Test 3 chunking configurations with the same naive retriever (k=10).
Goal: find the best chunk_size and overlap before experimenting with retrieval strategies.

| Config | chunk_size | chunk_overlap |
|--------|-----------|---------------|
| Small  | 250       | 50            |
| Medium | 500       | 50            |
| Large  | 1000      | 100           |

In [5]:
from rag.chunking import chunk_documents
from rag.vectorstore import create_vector_store

chunking_configs = [
    {"name": "small_250_50",   "chunk_size": 250,  "chunk_overlap": 50},
    {"name": "medium_500_50",  "chunk_size": 500,  "chunk_overlap": 50},
    {"name": "large_1000_100", "chunk_size": 1000, "chunk_overlap": 100},
]

chunking_results = {}

for config in chunking_configs:
    chunks = chunk_documents(docs, chunk_size=config["chunk_size"], chunk_overlap=config["chunk_overlap"])
    print(f"\nChunking '{config['name']}': {len(chunks)} chunks")

    vs = create_vector_store(chunks)
    retriever = vs.as_retriever(search_kwargs={"k": 10})

    result_df = run_experiment(
        retriever_fn=retriever.invoke,
        name=f"Naive k=10, chunks {config['name']}",
    )
    result_df.to_csv(os.path.join(RESULTS_DIR, f"chunking_{config['name']}.csv"), index=False)
    chunking_results[config["name"]] = result_df


Chunking 'small_250_50': 6309 chunks

Running: Naive k=10, chunks small_250_50
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[34]: TimeoutError()



  Results for Naive k=10, chunks small_250_50:
    context_recall                           0.6872  (21/21 valid)
    faithfulness                             0.8346  (21/21 valid)
    factual_correctness(mode=f1)             0.5781  (21/21 valid)
    answer_relevancy                         0.9487  (21/21 valid)
    context_entity_recall                    0.2613  (20/21 valid)

Chunking 'medium_500_50': 3073 chunks

Running: Naive k=10, chunks medium_500_50
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



  Results for Naive k=10, chunks medium_500_50:
    context_recall                           0.7605  (21/21 valid)
    faithfulness                             0.8333  (21/21 valid)
    factual_correctness(mode=f1)             0.5576  (21/21 valid)
    answer_relevancy                         0.9452  (21/21 valid)
    context_entity_recall                    0.1906  (21/21 valid)

Chunking 'large_1000_100': 1365 chunks

Running: Naive k=10, chunks large_1000_100
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[14]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[89]: TimeoutError


  Results for Naive k=10, chunks large_1000_100:
    context_recall                           0.8418  (21/21 valid)
    faithfulness                             0.8686  (21/21 valid)
    factual_correctness(mode=f1)             0.6181  (21/21 valid)
    answer_relevancy                         0.9468  (21/21 valid)
    context_entity_recall                    0.2276  (19/21 valid)


In [6]:
# Compare chunking results
chunking_comparison = pd.DataFrame({
    name: df.select_dtypes(include="number").mean().to_dict()
    for name, df in chunking_results.items()
}).T

print("=== Chunking Comparison (Naive k=10) ===")
chunking_comparison

=== Chunking Comparison (Naive k=10) ===


,context_recall,faithfulness,factual_correctness(mode=f1),answer_relevancy,context_entity_recall
small_250_50,0.687235,0.834649,0.578095,0.948685,0.261310
medium_500_50,0.760462,0.833282,0.557619,0.945158,0.190566
large_1000_100,0.841775,0.868562,0.618095,0.946808,0.227556


In [7]:
# Pick the best chunking based on composite of key metrics
KEY_METRICS = ["context_recall", "faithfulness", "factual_correctness", "answer_relevancy"]
available_metrics = [m for m in KEY_METRICS if m in chunking_comparison.columns]

chunking_comparison["composite_avg"] = chunking_comparison[available_metrics].mean(axis=1)
best_chunking = chunking_comparison["composite_avg"].idxmax()
print(f"Best chunking: {best_chunking}")
print(f"Score: {chunking_comparison.loc[best_chunking, 'composite_avg']:.4f}")
chunking_comparison[[*available_metrics, "composite_avg"]]

Best chunking: large_1000_100
Score: 0.8857


,context_recall,faithfulness,answer_relevancy,composite_avg
small_250_50,0.687235,0.834649,0.948685,0.823523
medium_500_50,0.760462,0.833282,0.945158,0.846300
large_1000_100,0.841775,0.868562,0.946808,0.885715


---
## 5.5 K-Value Experiment

Using the best chunking config, test whether retrieving fewer chunks improves
quality by reducing noise. We test k=3, 5, 10 on naive dense retrieval.
The winning k will be used for all retrieval strategy experiments.

In [8]:
# Build vector store with best chunking config for k experiments
best_config = next(c for c in chunking_configs if c["name"] == best_chunking)
print(f"Using best chunking: {best_config}")

best_chunks = chunk_documents(docs, chunk_size=best_config["chunk_size"], chunk_overlap=best_config["chunk_overlap"])
print(f"Created {len(best_chunks)} chunks")

vs = create_vector_store(best_chunks)

k_values = [3, 5, 10]
k_results = {}

for k in k_values:
    retriever_k = vs.as_retriever(search_kwargs={"k": k})
    result_df = run_experiment(retriever_k.invoke, f"Naive k={k}")
    result_df.to_csv(os.path.join(RESULTS_DIR, f"k_experiment_naive_k{k}.csv"), index=False)
    k_results[f"k={k}"] = result_df

k_comparison = pd.DataFrame({
    name: df.select_dtypes(include="number").mean().to_dict()
    for name, df in k_results.items()
}).T

composite_metrics = ["context_recall", "faithfulness", "answer_relevancy"]
available = [m for m in composite_metrics if m in k_comparison.columns]
k_comparison["composite_avg"] = k_comparison[available].mean(axis=1)

print("\n=== K-Value Comparison (Naive Dense) ===")
print(k_comparison[[*available, "composite_avg"]])

best_k_name = k_comparison["composite_avg"].idxmax()
best_k = int(best_k_name.split("=")[1])
print(f"\nBest k: {best_k} (composite avg: {k_comparison.loc[best_k_name, 'composite_avg']:.4f})")

k_comparison.to_csv(os.path.join(RESULTS_DIR, "k_value_comparison.csv"))

Using best chunking: {'name': 'large_1000_100', 'chunk_size': 1000, 'chunk_overlap': 100}
Created 1365 chunks

Running: Naive k=3
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



  Results for Naive k=3:
    context_recall                           0.6194  (21/21 valid)
    faithfulness                             0.7212  (21/21 valid)
    factual_correctness(mode=f1)             0.6062  (21/21 valid)
    answer_relevancy                         0.9459  (21/21 valid)
    context_entity_recall                    0.1879  (21/21 valid)

Running: Naive k=5
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



  Results for Naive k=5:
    context_recall                           0.7390  (21/21 valid)
    faithfulness                             0.7871  (21/21 valid)
    factual_correctness(mode=f1)             0.5919  (21/21 valid)
    answer_relevancy                         0.9511  (21/21 valid)
    context_entity_recall                    0.1830  (21/21 valid)

Running: Naive k=10
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[89]: TimeoutError()



  Results for Naive k=10:
    context_recall                           0.8259  (21/21 valid)
    faithfulness                             0.9062  (21/21 valid)
    factual_correctness(mode=f1)             0.6048  (21/21 valid)
    answer_relevancy                         0.9478  (21/21 valid)
    context_entity_recall                    0.1906  (20/21 valid)

=== K-Value Comparison (Naive Dense) ===
      context_recall  faithfulness  answer_relevancy  composite_avg
k=3         0.619393      0.721164          0.945875       0.762144
k=5         0.738992      0.787059          0.951056       0.825702
k=10        0.825902      0.906154          0.947803       0.893286

Best k: 10 (composite avg: 0.8933)


---
## 6. Retrieval Experiments

Using the best chunking and best k from above, test 5 retrieval strategies:
1. **Naive** — dense cosine similarity
2. **BM25** — keyword/term matching
3. **Rerank** — dense k=20 -> Cohere rerank to top 5
4. **Parent-child** — embed small chunks (400), retrieve parent chunks (2000)
5. **Ensemble** — all 4 retrievers with equal-weight reciprocal rank fusion

### 6.1 Naive Retriever (k=best_k)

In [9]:
naive_retriever = vs.as_retriever(search_kwargs={"k": best_k})
print(f"Naive retriever with k={best_k}")

# Reuse k experiment result if best_k was already tested
best_k_key = f"k={best_k}"
if best_k_key in k_results:
    naive_result = k_results[best_k_key]
    print(f"Reusing k experiment result for naive k={best_k}")
    for col in naive_result.select_dtypes(include="number").columns:
        valid = naive_result[col].dropna()
        print(f"  {col:40s} {valid.mean():.4f}")
else:
    naive_result = run_experiment(naive_retriever.invoke, f"Naive k={best_k}")

naive_result.to_csv(os.path.join(RESULTS_DIR, "retrieval_naive_dense.csv"), index=False)

Naive retriever with k=10
Reusing k experiment result for naive k=10
  context_recall                           0.8259
  faithfulness                             0.9062
  factual_correctness(mode=f1)             0.6048
  answer_relevancy                         0.9478
  context_entity_recall                    0.1906


### 6.2 BM25 Retriever

In [10]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(best_chunks, k=best_k)

# Sanity check
test_docs = bm25_retriever.invoke("What colors suit a Deep Autumn?")
print(f"BM25 returned {len(test_docs)} docs (k={best_k})")
print(f"Top result: {test_docs[0].page_content[:100]}...")

bm25_result = run_experiment(bm25_retriever.invoke, f"BM25 k={best_k}")
bm25_result.to_csv(os.path.join(RESULTS_DIR, "retrieval_bm25.csv"), index=False)

BM25 returned 10 docs (k=10)
Top result: **Value** refers to the lightness or darkness of your coloring, determined by the amount of white or...

Running: BM25 k=10
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



  Results for BM25 k=10:
    context_recall                           0.6661  (21/21 valid)
    faithfulness                             0.7676  (21/21 valid)
    factual_correctness(mode=f1)             0.5805  (21/21 valid)
    answer_relevancy                         0.9089  (21/21 valid)
    context_entity_recall                    0.1640  (21/21 valid)


### 6.3 Rerank (Contextual Compression)

Retrieve k=20 candidates with dense retrieval, then rerank with Cohere to top 5.

In [13]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

wide_retriever = vs.as_retriever(search_kwargs={"k": 20})
cohere_reranker = CohereRerank(model="rerank-v3.5", top_n=best_k)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=cohere_reranker,
    base_retriever=wide_retriever,
)

# Sanity check
test_docs = rerank_retriever.invoke("What colors suit a Deep Autumn?")
print(f"Rerank returned {len(test_docs)} docs (from 20 candidates)")
print(f"Top result: {test_docs[0].page_content[:100]}...")

# Cohere trial key: 10 calls/min, sleep 7s between calls to avoid rate limit
rerank_result = run_experiment(rerank_retriever.invoke, f"Rerank (k=20 -> top {best_k})", sleep_between=7)
rerank_result.to_csv(os.path.join(RESULTS_DIR, "retrieval_rerank.csv"), index=False)

Rerank returned 10 docs (from 20 candidates)
Top result: **Best Colors for Deep Autumn:** Deep Autumn thrives in dark, warm, saturated colors. Excellent choi...

Running: Rerank (k=20 -> top 10)
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



  Results for Rerank (k=20 -> top 10):
    context_recall                           0.8405  (21/21 valid)
    faithfulness                             0.8480  (21/21 valid)
    factual_correctness(mode=f1)             0.5610  (21/21 valid)
    answer_relevancy                         0.9454  (21/21 valid)
    context_entity_recall                    0.2758  (21/21 valid)


### 6.4 Parent-Child Retriever

Embed small child chunks (400 chars) for precise matching,
but return the larger parent chunks (2000 chars) for richer context.

In [14]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_qdrant import QdrantVectorStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

parent_child_vs = QdrantVectorStore.from_documents(
    [],
    embeddings,
    location=":memory:",
    collection_name="parent_child",
)

docstore = InMemoryStore()
parent_child_retriever = ParentDocumentRetriever(
    vectorstore=parent_child_vs,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

parent_child_retriever.add_documents(docs)
print(f"Parent-child retriever ready")

# Sanity check
test_docs = parent_child_retriever.invoke("What colors suit a Deep Autumn?")
print(f"Parent-child returned {len(test_docs)} docs")
print(f"Top result length: {len(test_docs[0].page_content)} chars")
print(f"Top result: {test_docs[0].page_content[:100]}...")

parent_child_result = run_experiment(parent_child_retriever.invoke, "Parent-Child")
parent_child_result.to_csv(os.path.join(RESULTS_DIR, "retrieval_parent_child.csv"), index=False)

Parent-child retriever ready
Parent-child returned 3 docs
Top result length: 1929 chars
Top result: **Colors to Avoid:** Avoid cool-toned colors like cool pink, cool grey, navy, and pure white. Soft, ...

Running: Parent-Child
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



  Results for Parent-Child:
    context_recall                           0.8022  (21/21 valid)
    faithfulness                             0.8859  (21/21 valid)
    factual_correctness(mode=f1)             0.5838  (21/21 valid)
    answer_relevancy                         0.9465  (21/21 valid)
    context_entity_recall                    0.2432  (21/21 valid)


### 6.5 Ensemble Retriever

Combines all retrieval strategies with equal weighting using reciprocal rank fusion,
following the pattern from class (Session 11).

In [15]:
from langchain_classic.retrievers import EnsembleRetriever

retriever_list = [naive_retriever, bm25_retriever, rerank_retriever, parent_child_retriever]
equal_weighting = [1 / len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list,
    weights=equal_weighting,
)

# Sanity check
test_docs = ensemble_retriever.invoke("What colors suit a Deep Autumn?")
print(f"Ensemble returned {len(test_docs)} docs (from {len(retriever_list)} retrievers)")
print(f"Top result: {test_docs[0].page_content[:100]}...")

# sleep_between=7 because ensemble includes the rerank retriever (Cohere rate limit)
ensemble_result = run_experiment(ensemble_retriever.invoke, "Ensemble (all 4 retrievers)", sleep_between=7)
ensemble_result.to_csv(os.path.join(RESULTS_DIR, "retrieval_ensemble.csv"), index=False)

Ensemble returned 22 docs (from 4 retrievers)
Top result: **Best Colors for Deep Autumn:** Deep Autumn thrives in dark, warm, saturated colors. Excellent choi...

Running: Ensemble (all 4 retrievers)
  Processed 7/21 queries
  Processed 14/21 queries
  Processed 21/21 queries
  Done with 21 queries. Evaluating with RAGAS...


Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[24]: TimeoutError()
Exception raised in Job[84]: TimeoutError()



  Results for Ensemble (all 4 retrievers):
    context_recall                           0.9405  (21/21 valid)
    faithfulness                             0.9378  (21/21 valid)
    factual_correctness(mode=f1)             0.6290  (21/21 valid)
    answer_relevancy                         0.9473  (21/21 valid)
    context_entity_recall                    0.2806  (19/21 valid)


---
## 7. Final Comparison

Compare all retrieval strategies across RAGAS metrics.

In [17]:
all_results = {
    f"Naive k={best_k}": naive_result,
    f"BM25 k={best_k}": bm25_result,
    f"Rerank (k=20->{best_k})": rerank_result,
    "Parent-Child": parent_child_result,
    "Ensemble": ensemble_result,
}

comparison = pd.DataFrame({
    name: df.select_dtypes(include="number").mean().to_dict()
    for name, df in all_results.items()
}).T

print("=" * 80)
print("RETRIEVAL STRATEGY COMPARISON")
print("=" * 80)
comparison

RETRIEVAL STRATEGY COMPARISON


,context_recall,faithfulness,factual_correctness(mode=f1),answer_relevancy,context_entity_recall
Naive k=10,0.825902,0.906154,0.604762,0.947803,0.190620
BM25 k=10,0.666131,0.767561,0.580476,0.908873,0.163953
Rerank (k=20->10),0.840548,0.847987,0.560952,0.945405,0.275787
Parent-Child,0.802206,0.885930,0.583810,0.946519,0.243190
Ensemble,0.940548,0.937844,0.629048,0.947332,0.280626


In [18]:
# Highlight the best strategy per metric
print("Best strategy per metric:")
for col in comparison.columns:
    best = comparison[col].idxmax()
    print(f"  {col:40s} -> {best} ({comparison.loc[best, col]:.4f})")

# Overall best (composite of context_recall, faithfulness, answer_relevancy)
composite_metrics = ["context_recall", "faithfulness", "answer_relevancy"]
available = [m for m in composite_metrics if m in comparison.columns]
comparison["composite_avg"] = comparison[available].mean(axis=1)
overall_best = comparison["composite_avg"].idxmax()
print(f"\nOverall best: {overall_best} (composite avg: {comparison.loc[overall_best, 'composite_avg']:.4f})")
comparison

Best strategy per metric:
  context_recall                           -> Ensemble (0.9405)
  faithfulness                             -> Ensemble (0.9378)
  factual_correctness(mode=f1)             -> Ensemble (0.6290)
  answer_relevancy                         -> Naive k=10 (0.9478)
  context_entity_recall                    -> Ensemble (0.2806)

Overall best: Ensemble (composite avg: 0.9419)


,context_recall,faithfulness,factual_correctness(mode=f1),answer_relevancy,context_entity_recall,composite_avg
Naive k=10,0.825902,0.906154,0.604762,0.947803,0.190620,0.893286
BM25 k=10,0.666131,0.767561,0.580476,0.908873,0.163953,0.780855
Rerank (k=20->10),0.840548,0.847987,0.560952,0.945405,0.275787,0.877980
Parent-Child,0.802206,0.885930,0.583810,0.946519,0.243190,0.878218
Ensemble,0.940548,0.937844,0.629048,0.947332,0.280626,0.941908


---
## 8. LangSmith Cost & Latency Comparison

Upload test samples to a LangSmith dataset, then run each retriever through
LangSmith evaluation to compare cost and latency in the dashboard.

In [19]:
import uuid
from langsmith import Client

langsmith_client = Client()
dataset_name = f"Stylists AI - Retrieval Eval - {uuid.uuid4().hex[:8]}"

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description="Stylists.ai retrieval strategy comparison"
)

# Upload test samples to the dataset
for _, row in test_df.iterrows():
    langsmith_client.create_example(
        inputs={"question": row["user_input"]},
        outputs={"answer": row["reference"]},
        dataset_id=langsmith_dataset.id,
    )

print(f"Created LangSmith dataset: {dataset_name}")
print(f"Uploaded {len(test_df)} examples")

Created LangSmith dataset: Stylists AI - Retrieval Eval - 39dee111
Uploaded 21 examples


In [21]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate as ls_evaluate

qa_evaluator = create_llm_as_judge(
    prompt=(
        "You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\n"
        "Input: {inputs}\n"
        "Prediction: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Is the prediction correct? Return 1 if correct, 0 if incorrect."
    ),
    feedback_key="qa",
    model="openai:gpt-4.1-mini",
)

labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4.1-mini",
)


def make_retriever_target(retriever_fn, sleep_between=0):
    """Wrap a retriever function into a RAG target for LangSmith evaluation."""
    def target(inputs):
        question = inputs["question"]
        if sleep_between > 0:
            time.sleep(sleep_between)
        retrieved = retriever_fn(question)
        docs_content = "\n\n".join(doc.page_content for doc in retrieved)
        messages = rag_prompt.format_messages(question=question, context=docs_content)
        response = rag_llm.invoke(messages).content
        return response
    return target


retriever_targets = {
    "naive": (naive_retriever.invoke, 0),
    "bm25": (bm25_retriever.invoke, 0),
    "rerank": (rerank_retriever.invoke, 7),
    "parent_child": (parent_child_retriever.invoke, 0),
    "ensemble": (ensemble_retriever.invoke, 7),
}

for name, (retriever_fn, sleep) in retriever_targets.items():
    print(f"\nRunning LangSmith evaluation for: {name}")
    ls_evaluate(
        make_retriever_target(retriever_fn, sleep_between=sleep),
        data=dataset_name,
        evaluators=[qa_evaluator, labeled_helpfulness_evaluator],
        metadata={"revision_id": name},
        experiment_prefix=name,
    )


Running LangSmith evaluation for: naive
View the evaluation results for experiment: 'naive-1ce443b7' at:
https://smith.langchain.com/o/991456bb-c3e2-422e-957d-4e5bfe176b65/datasets/4f5e3367-02f5-4521-b935-06b753ca34e1/compare?selectedSessions=5a9af3a1-8e7f-4a68-a39b-50f8370bde81




0it [00:00, ?it/s]


Running LangSmith evaluation for: bm25
View the evaluation results for experiment: 'bm25-a6b7ebab' at:
https://smith.langchain.com/o/991456bb-c3e2-422e-957d-4e5bfe176b65/datasets/4f5e3367-02f5-4521-b935-06b753ca34e1/compare?selectedSessions=214b3ce1-e3ef-4bc7-a532-7b7940d21745




0it [00:00, ?it/s]


Running LangSmith evaluation for: rerank
View the evaluation results for experiment: 'rerank-dafb1dba' at:
https://smith.langchain.com/o/991456bb-c3e2-422e-957d-4e5bfe176b65/datasets/4f5e3367-02f5-4521-b935-06b753ca34e1/compare?selectedSessions=cb28ae69-777b-40da-815f-ff2ee49f7237




0it [00:00, ?it/s]


Running LangSmith evaluation for: parent_child
View the evaluation results for experiment: 'parent_child-95cbb9ca' at:
https://smith.langchain.com/o/991456bb-c3e2-422e-957d-4e5bfe176b65/datasets/4f5e3367-02f5-4521-b935-06b753ca34e1/compare?selectedSessions=097102c9-690c-4fbf-a5c9-b1a7d07c5baf




0it [00:00, ?it/s]


Running LangSmith evaluation for: ensemble
View the evaluation results for experiment: 'ensemble-6d1c01d6' at:
https://smith.langchain.com/o/991456bb-c3e2-422e-957d-4e5bfe176b65/datasets/4f5e3367-02f5-4521-b935-06b753ca34e1/compare?selectedSessions=8e9da49c-7528-4e15-9f09-12d81b18ce84




0it [00:00, ?it/s]

## 9. Save Results

In [23]:
# Save all comparison results
comparison.to_csv(os.path.join(RESULTS_DIR, "retrieval_comparison.csv"))
chunking_comparison.to_csv(os.path.join(RESULTS_DIR, "chunking_comparison.csv"))

print("Saved retrieval_comparison.csv")
print("Saved chunking_comparison.csv")

# Summary of all saved files
print("\nAll result files:")
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.endswith(".csv"):
        print(f"  {f}")

Saved retrieval_comparison.csv
Saved chunking_comparison.csv

All result files:
  baseline_per_query.csv
  baseline_responses.csv
  chunking_comparison.csv
  chunking_large_1000_100.csv
  chunking_medium_500_50.csv
  chunking_small_250_50.csv
  k_experiment_naive_k10.csv
  k_experiment_naive_k3.csv
  k_experiment_naive_k5.csv
  k_value_comparison.csv
  retrieval_bm25.csv
  retrieval_comparison.csv
  retrieval_ensemble.csv
  retrieval_naive_dense.csv
  retrieval_parent_child.csv
  retrieval_rerank.csv


---
## 10. Conclusion

### LangSmith Cost & Latency Dashboard

[View the public LangSmith dataset](https://smith.langchain.com/public/798c2f33-3244-4de4-acf7-9f9bfbc4358d/d)

![LangSmith Retriever Comparison](results/langsmith-retriever-comparison.png)

### Combined Performance, Latency & Cost Comparison

| Strategy | Composite Avg | Context Recall | Faithfulness | Answer Relevancy | P50 Latency | Cost | LLM QA Score |
|---|---|---|---|---|---|---|---|
| **Ensemble** | **0.942** | **0.941** | **0.938** | 0.947 | 15.2s | ~$0.05 | 1.00 |
| **Naive (k=10)** | **0.893** | 0.826 | 0.906 | **0.948** | 6.3s | ~$0.01 | 0.95 |
| Rerank (k=20->10) | 0.878 | 0.841 | 0.848 | 0.945 | 13.8s | ~$0.02 | 0.90 |
| Parent-Child | 0.878 | 0.802 | 0.886 | 0.947 | 4.9s | ~$0.02 | 1.00 |
| BM25 (k=10) | 0.781 | 0.666 | 0.768 | 0.909 | 5.7s | ~$0.01 | 0.95 |

### Key Findings

1. **Ensemble is the best-performing retriever** across all RAGAS metrics, achieving the highest composite average (0.942) with top scores in context recall (0.941), faithfulness (0.938), and factual correctness (0.629).

2. **Naive dense retrieval is the best tradeoff for production.** It achieves the second-highest composite (0.893) at a fraction of the cost (~$0.01 vs ~$0.05) and latency (6.3s vs 15.2s). It requires no external API dependencies (no Cohere key), making it simpler to deploy and maintain.

3. **BM25 keyword search underperforms** on this fashion/styling corpus (composite 0.781). Styling queries require semantic understanding that lexical matching cannot capture.

4. **Rerank and Parent-Child tie** in composite (0.878) but through different strengths — rerank finds more relevant context, while parent-child reduces hallucination with larger coherent chunks.

### Decision

**We will use naive dense retrieval (k=10, chunk size 1000/100) for the production Stylists.ai app**, balancing strong retrieval quality with minimal latency, cost, and operational complexity. Ensemble remains a viable upgrade path if quality requirements increase.